# Type VI

In [17]:
import os
from pathlib import Path
import re

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

cwd = os.getcwd()
if cwd.endswith('notebook'):
    os.chdir('..')
    cwd = os.getcwd()

In [2]:
sns.set_palette('colorblind')
sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.8)
plt.rcParams['font.family'] = 'Helvetica'

palette = sns.color_palette().as_hex()

data_folder = Path('./data')
assert data_folder.is_dir()

figures_folder = Path('./figures')
assert figures_folder.is_dir()

In [5]:
gtdb_metadata = pd.read_csv(data_folder / 'gtdb_metadata.csv', index_col='ncbi_accession')
gtdb_metadata.head()

archaeal_accessions = sorted(gtdb_metadata[gtdb_metadata['domain'] == 'Archaea'].index)

In [25]:
pgh_df = pd.read_csv(data_folder / 'pgh_proteins.csv')
archaeal_pgh = pgh_df[pgh_df['domain'] == 'Archaea'].reset_index(drop=True)
archaeal_pgh['id'] = archaeal_pgh.apply(lambda row: f"{row['protein_id']}@{row['assembly_accession']}", axis=1)
archaeal_pgh.head()

,assembly_accession,domain,gtdb_phylum,gtdb_class,gtdb_order,gtdb_family,gtdb_genus,gtdb_species,ncbi_organism_name,protein_id,pgh_architecture,id
0,GCA_003663345.1,Archaea,Aenigmatarchaeota,Aenigmatarchaeia,PWEA01,B50-G16,B50-G16,B50-G16 sp003663345,Candidatus Aenigmarchaeota archaeon,RLJ01288.1,PG_binding_3+Glyco_hydro_108,RLJ01288.1@GCA_003663345.1
1,GCA_018304545.1,Archaea,Aenigmatarchaeota,Aenigmatarchaeia,CG10238-14,CG10238-14,JAGVVZ01,JAGVVZ01 sp018304545,Candidatus Aenigmarchaeota archaeon,MBS3052716.1,LysM+Amidase_2,MBS3052716.1@GCA_018304545.1
2,GCA_025058005.1,Archaea,Aenigmatarchaeota,Aenigmatarchaeia,CG10238-14,CG10238-14,JAHLMN01,JAHLMN01 sp025058005,Candidatus Aenigmarchaeota archaeon,MCS7135447.1,LysM+NLPC_P60,MCS7135447.1@GCA_025058005.1
3,GCA_015661515.1,Archaea,Aenigmatarchaeota,Aenigmatarchaeia,Aenigmatarchaeales,SZUA-1535,SZUA-1535,SZUA-1535 sp015661515,Nanoarchaeota archaeon,HIQ49991.1,PG_binding_3+Glyco_hydro_108,HIQ49991.1@GCA_015661515.1
4,GCA_902384455.1,Archaea,Altiarchaeota,Altiarchaeia,IMC4,SCGC-AAA252-I15,CABMCB01,CABMCB01 sp902384455,uncultured archaeon,VVB52780.1,PG_binding_3+Glyco_hydro_108,VVB52780.1@GCA_902384455.1


In [83]:
catalytic_only = pd.read_csv(data_folder / 'catalytic_only_archaeal_proteins.csv')
catalytic_only = catalytic_only[catalytic_only['hmm_query'] != 'Amidase'].reset_index(drop=True)

m23_only = catalytic_only[catalytic_only['hmm_query'] == 'Peptidase_M23'].reset_index(drop=True)

catalytic_only.head()

,assembly_accession,protein_id,hmm_accession,hmm_query,domain,gtdb_phylum,gtdb_class,gtdb_order,gtdb_family,gtdb_genus,gtdb_species,id
0,GCA_000220355.1,EGQ39878.1,PF17676.4,Peptidase_S66C,Archaea,Nanohaloarchaeota,Nanosalinia,Nanosalinales,Nanosalinaceae,Nanosalinarum,Nanosalinarum sp000220355,EGQ39878.1@GCA_000220355.1
1,GCA_000220355.1,EGQ39880.1,PF02016.18,Peptidase_S66,Archaea,Nanohaloarchaeota,Nanosalinia,Nanosalinales,Nanosalinaceae,Nanosalinarum,Nanosalinarum sp000220355,EGQ39880.1@GCA_000220355.1
2,GCA_000220355.1,EGQ39880.1,PF17676.4,Peptidase_S66C,Archaea,Nanohaloarchaeota,Nanosalinia,Nanosalinales,Nanosalinaceae,Nanosalinarum,Nanosalinarum sp000220355,EGQ39880.1@GCA_000220355.1
3,GCA_000220355.1,EGQ39881.1,PF02016.18,Peptidase_S66,Archaea,Nanohaloarchaeota,Nanosalinia,Nanosalinales,Nanosalinaceae,Nanosalinarum,Nanosalinarum sp000220355,EGQ39881.1@GCA_000220355.1
4,GCA_000220355.1,EGQ39881.1,PF17676.4,Peptidase_S66C,Archaea,Nanohaloarchaeota,Nanosalinia,Nanosalinales,Nanosalinaceae,Nanosalinarum,Nanosalinarum sp000220355,EGQ39881.1@GCA_000220355.1


In [78]:
signalp_output = pd.read_csv(data_folder / 'signalp6_output.csv').rename(columns={
    'protein_id': 'id',
}).set_index('id')
signalp_output['assembly_accession'] = [p.split('@')[1] if len(p.split('@')) > 1 else None for p in signalp_output.index]
signalp_output.head()

,signal_peptide,probability,cleavage_site,cleavage_site_prob,assembly_accession
id,,,,,
BAC24414.1@GCA_000008885.1,Sec/SPII,0.987955,24-25,0.966853,GCA_000008885.1
BAF58967.1@GCA_000010565.1,Sec/SPII,0.685612,22-23,0.602861,GCA_000010565.1
BAF61015.1@GCA_000010565.1,Sec/SPI,0.998938,33-34,0.969633,GCA_000010565.1
BAG83382.1@GCA_000010645.1,Sec/SPI,0.786221,29-30,0.700540,GCA_000010645.1
ABG39042.1@GCA_000014225.1,Sec/SPI,0.868330,44-45,0.823325,GCA_000014225.1


In [4]:
pfam_path = Path('../data/gtdb_r214.1/db_prokaryotes/all_proteins_Pfam-A_hits.csv.gz')
pfam_path.is_file()

True

## Type VI secretion systems

As per [Zachs et al., 2024](https://doi.org/10.1126/sciadv.adp7088), `Phage_sheath_1` (PF04984) is a key component of Contractile Injection Systems (CIS) in archaea.

In [45]:
pfam_iterator = pd.read_csv(pfam_path, chunksize=int(1e5), index_col='id')

dfs = []
for chunk_df in pfam_iterator:
    relevant_df = chunk_df[chunk_df['assembly_accession'].isin(archaeal_accessions)]
    relevant_df = relevant_df[relevant_df['hmm_query'] == 'Phage_sheath_1']

    if len(relevant_df) > 0:
        dfs.append(relevant_df)

type6_summary_df = pd.concat(dfs)
type6_summary_df = pd.merge(
    type6_summary_df,
    gtdb_metadata.reset_index()[
        ['ncbi_accession', 'domain', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus', 'gtdb_species']
    ].rename(columns={'ncbi_accession': 'assembly_accession'}),
    on='assembly_accession',
    how='left',
)
type6_summary_df['id'] = type6_summary_df.apply(lambda row: f"{row['protein_id']}@{row['assembly_accession']}", axis=1)

type6_summary_df = type6_summary_df.drop_duplicates('id')

type6_summary_df = pd.merge(
    type6_summary_df,
    archaeal_pgh.drop_duplicates('assembly_accession')[['assembly_accession', 'pgh_architecture']],
    on='assembly_accession',
    how='left',
)

type6_summary_df = pd.merge(
    type6_summary_df,
    catalytic_only.drop_duplicates('assembly_accession')[['assembly_accession', 'hmm_query']].rename(columns={
        'hmm_query': 'catalytic_only'
    }),
    on='assembly_accession',
    how='left',
)
type6_summary_df['has_catalytic_only'] = type6_summary_df['catalytic_only'].notnull().astype(int)
type6_summary_df = type6_summary_df.drop(columns=['catalytic_only'])

type6_summary_df.head()

,assembly_accession,protein_id,hmm_accession,hmm_query,evalue,bitscore,accuracy,start,end,domain,gtdb_phylum,gtdb_class,gtdb_order,gtdb_family,gtdb_genus,gtdb_species,id,pgh_architecture,has_catalytic_only
0,GCA_000337115.1,ELZ08799.1,PF04984.17,Phage_sheath_1,9.600000e-31,104.7,0.89,225,397,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Natrialbaceae,Natrinema,Natrinema thermotolerans,ELZ08799.1@GCA_000337115.1,NaN,1
1,GCA_000416085.1,ERH11266.1,PF04984.17,Phage_sheath_1,5.600000e-30,101.9,0.90,165,332,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Haloarculaceae,Halovenus,Halovenus sp000416085,ERH11266.1@GCA_000416085.1,NaN,1
2,GCA_001412355.1,KQC04758.1,PF04984.17,Phage_sheath_1,1.500000e-29,100.0,0.96,239,416,Archaea,Halobacteriota,Methanomicrobia,Methanomicrobiales,Methanomicrobiaceae,LKUD01,LKUD01 sp001412355,KQC04758.1@GCA_001412355.1,NaN,1
3,GCA_001563795.1,LKML01000027.1_16,PF04984.17,Phage_sheath_1,9.600000e-31,104.2,0.91,285,455,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Haloferacaceae,Halohasta,Halohasta sp001563795,LKML01000027.1_16@GCA_001563795.1,NaN,0
4,GCA_001563885.1,LKMM01000005.1_8,PF04984.17,Phage_sheath_1,5.900000e-27,91.6,0.89,301,468,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Natrialbaceae,Natrarchaeobaculum,Natrarchaeobaculum sp001563885,LKMM01000005.1_8@GCA_001563885.1,NaN,0


In [46]:
type6_stats = type6_summary_df[
    ['gtdb_phylum', 'assembly_accession', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus']
].groupby(
    ['gtdb_phylum']
).nunique().sort_values('assembly_accession', ascending=False)

type6_stats['percent'] = type6_stats.apply(
    lambda row: np.round(
        100 * row['assembly_accession'] / len(gtdb_metadata[gtdb_metadata['gtdb_phylum'] == row.name]),
        1
    ),
    axis=1,
)

type6_stats = type6_stats.rename(columns={
    'assembly_accession': '# genomes',
    'percent': '% genomes',
    'gtdb_class': '# GTDB class',
    'gtdb_order': '# GTDB order',
    'gtdb_family': '# GTDB family',
    'gtdb_genus': '# GTDB genus',
})
type6_stats

,# genomes,# GTDB class,# GTDB order,# GTDB family,# GTDB genus,% genomes
gtdb_phylum,,,,,,
Halobacteriota,96,5,6,19,56,13.0
Thermoproteota,31,4,9,17,23,4.4
Thermoplasmatota,13,3,5,7,10,2.2
Nanoarchaeota,5,1,2,2,4,0.7
Asgardarchaeota,4,3,3,3,3,2.1
Methanobacteriota,3,1,1,1,3,2.0
Aenigmatarchaeota,2,1,2,2,2,1.5
Altiarchaeota,1,1,1,1,1,3.8


In [47]:
df = type6_summary_df.copy()
df['has_pgh'] = df['pgh_architecture'].notnull().astype(int)
pgh_stats = df[['assembly_accession', 'gtdb_phylum', 'has_pgh']].drop_duplicates('assembly_accession')[['gtdb_phylum', 'has_pgh']].groupby('gtdb_phylum').sum()

type6_stats_2 = pd.merge(
    type6_stats,
    pgh_stats,
    on='gtdb_phylum',
    how='inner',
).rename(columns={
    'has_pgh': '# PGH',
})
type6_stats_2

,# genomes,# GTDB class,# GTDB order,# GTDB family,# GTDB genus,% genomes,# PGH
gtdb_phylum,,,,,,,
Halobacteriota,96,5,6,19,56,13.0,10
Thermoproteota,31,4,9,17,23,4.4,2
Thermoplasmatota,13,3,5,7,10,2.2,0
Nanoarchaeota,5,1,2,2,4,0.7,3
Asgardarchaeota,4,3,3,3,3,2.1,1
Methanobacteriota,3,1,1,1,3,2.0,0
Aenigmatarchaeota,2,1,2,2,2,1.5,0
Altiarchaeota,1,1,1,1,1,3.8,0


In [48]:
df = type6_summary_df.copy()
cat_only_stats = df[
    ['assembly_accession', 'gtdb_phylum', 'has_catalytic_only']
].drop_duplicates('assembly_accession')[['gtdb_phylum', 'has_catalytic_only']].groupby('gtdb_phylum').sum()

type6_stats_3 = pd.merge(
    type6_stats_2,
    cat_only_stats,
    on='gtdb_phylum',
    how='inner',
).rename(columns={
    'has_catalytic_only': '# catalytic only',
})
type6_stats_3

,# genomes,# GTDB class,# GTDB order,# GTDB family,# GTDB genus,% genomes,# PGH,# catalytic only
gtdb_phylum,,,,,,,,
Halobacteriota,96,5,6,19,56,13.0,10,49
Thermoproteota,31,4,9,17,23,4.4,2,15
Thermoplasmatota,13,3,5,7,10,2.2,0,13
Nanoarchaeota,5,1,2,2,4,0.7,3,5
Asgardarchaeota,4,3,3,3,3,2.1,1,4
Methanobacteriota,3,1,1,1,3,2.0,0,1
Aenigmatarchaeota,2,1,2,2,2,1.5,0,2
Altiarchaeota,1,1,1,1,1,3.8,0,0


In [61]:
type6_accessions = set(type6_summary_df['assembly_accession'].unique())
archaeal_pgh_accessions = set(archaeal_pgh['assembly_accession'].unique())

n_no_type6 = len(set(archaeal_accessions) - type6_accessions)

n_pgh_type6 = len(archaeal_pgh_accessions & type6_accessions)
n_pgh_no_type6 = len(archaeal_pgh_accessions - type6_accessions)

print(f'# genomes with CIS and PGH: {n_pgh_type6:,} of {len(type6_accessions):,} ({100 * n_pgh_type6 / len(type6_accessions):.0f}%)')
print(f'# genomes without CIS but with PGH: {n_pgh_no_type6:,} of {n_no_type6:,} ({100 * n_pgh_no_type6 / n_no_type6:.0f}%)')

# genomes with CIS and PGH: 16 of 155 (10%)
# genomes without CIS but with PGH: 159 of 3,551 (4%)


In [105]:
archaeal_pgh_accessions = set(archaeal_pgh['assembly_accession'].unique())
accessions_pgh_type6 = type6_summary_df[type6_summary_df['pgh_architecture'].notnull()]['assembly_accession'].unique()
type6_accessions = set(type6_summary_df['assembly_accession'].unique())

n = len(signalp_output[signalp_output['assembly_accession'].isin(accessions_pgh_type6)])
total = len(archaeal_pgh[archaeal_pgh['assembly_accession'].isin(accessions_pgh_type6)])

pgh_no_type6 = sorted(archaeal_pgh_accessions - type6_accessions)
n2 = len(signalp_output[signalp_output['assembly_accession'].isin(pgh_no_type6)])
total2 = len(archaeal_pgh[archaeal_pgh['assembly_accession'].isin(pgh_no_type6)])

print(f'Number of PGH proteins from CIS-containing genomes with a signal peptide: {n:,} of {total:,} ({100 * n / total:.1f}%)')
print(f'Number of PGH proteins from non-CIS-containing genomes with a signal peptide: {n2:,} of {total2:,} ({100 * n2 / total2:.1f}%)')

Number of PGH proteins from CIS-containing genomes with a signal peptide: 3 of 25 (12.0%)
Number of PGH proteins from non-CIS-containing genomes with a signal peptide: 59 of 185 (31.9%)


In [62]:
type6_accessions = set(type6_summary_df['assembly_accession'].unique())
archaeal_cat_accessions = set(catalytic_only['assembly_accession'].unique())

n_no_type6 = len(set(archaeal_accessions) - type6_accessions)

n_pgh_type6 = len(archaeal_cat_accessions & type6_accessions)
n_pgh_no_type6 = len(archaeal_cat_accessions - type6_accessions)

print(f'# genomes with CIS and cat only: {n_pgh_type6:,} of {len(type6_accessions):,} ({100 * n_pgh_type6 / len(type6_accessions):.0f}%)')
print(f'# genomes without CIS but with cat only: {n_pgh_no_type6:,} of {n_no_type6:,} ({100 * n_pgh_no_type6 / n_no_type6:.0f}%)')

# genomes with CIS and cat only: 89 of 155 (57%)
# genomes without CIS but with cat only: 1,943 of 3,551 (55%)


## FemAB

In [49]:
pfam_iterator = pd.read_csv(pfam_path, chunksize=int(1e5), index_col='id')

dfs = []
for chunk_df in pfam_iterator:
    relevant_df = chunk_df[chunk_df['assembly_accession'].isin(archaeal_accessions)]
    relevant_df = relevant_df[relevant_df['hmm_query'] == 'FemAB']

    if len(relevant_df) > 0:
        dfs.append(relevant_df)


summary_df = pd.concat(dfs)
summary_df = pd.merge(
    summary_df,
    gtdb_metadata.reset_index()[
        ['ncbi_accession', 'domain', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus', 'gtdb_species']
    ].rename(columns={'ncbi_accession': 'assembly_accession'}),
    on='assembly_accession',
    how='left',
)
summary_df['id'] = summary_df.apply(lambda row: f"{row['protein_id']}@{row['assembly_accession']}", axis=1)

summary_df = summary_df.drop_duplicates('id')

summary_df = pd.merge(
    summary_df,
    archaeal_pgh.drop_duplicates('assembly_accession')[['assembly_accession', 'pgh_architecture']],
    on='assembly_accession',
    how='left',
)

summary_df = pd.merge(
    summary_df,
    catalytic_only.drop_duplicates('assembly_accession')[['assembly_accession', 'hmm_query']].rename(columns={
        'hmm_query': 'catalytic_only'
    }),
    on='assembly_accession',
    how='left',
)
summary_df['has_catalytic_only'] = summary_df['catalytic_only'].notnull().astype(int)
summary_df = summary_df.drop(columns=['catalytic_only'])

summary_df.head()

,assembly_accession,protein_id,hmm_accession,hmm_query,evalue,bitscore,accuracy,start,end,domain,gtdb_phylum,gtdb_class,gtdb_order,gtdb_family,gtdb_genus,gtdb_species,id,pgh_architecture,has_catalytic_only
0,GCA_000306725.1,AFV22949.1,PF02388.19,FemAB,7.100000e-09,32.6,0.87,127,249,Archaea,Halobacteriota,Methanosarcinia,Methanosarcinales,Methanosarcinaceae,Methanolobus,Methanolobus psychrophilus,AFV22949.1@GCA_000306725.1,NaN,1
1,GCA_001412355.1,KQC03728.1,PF02388.19,FemAB,3.100000e-07,26.5,0.88,120,243,Archaea,Halobacteriota,Methanomicrobia,Methanomicrobiales,Methanomicrobiaceae,LKUD01,LKUD01 sp001412355,KQC03728.1@GCA_001412355.1,NaN,1
2,GCA_001512375.1,LFRN01000217.1_7,PF02388.19,FemAB,7.800000e-09,32.0,0.85,140,252,Archaea,Halobacteriota,Methanomicrobia,Methanomicrobiales,Methanoculleaceae,Methanoculleus,Methanoculleus thermohydrogenotrophicum,LFRN01000217.1_7@GCA_001512375.1,NaN,0
3,GCA_001563885.1,LKMM01000032.1_8,PF02388.19,FemAB,2.100000e-09,33.7,0.89,252,338,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Natrialbaceae,Natrarchaeobaculum,Natrarchaeobaculum sp001563885,LKMM01000032.1_8@GCA_001563885.1,NaN,0
4,GCA_001766825.1,OFV65888.1,PF02388.19,FemAB,3.200000e-07,26.1,0.84,143,243,Archaea,Halobacteriota,Syntropharchaeia,Syntropharchaeales,Syntropharchaeaceae,Syntropharchaeum,Syntropharchaeum butanivorans,OFV65888.1@GCA_001766825.1,NaN,0


In [50]:
femAB_stats = summary_df[
    ['gtdb_phylum', 'assembly_accession', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus']
].groupby(
    ['gtdb_phylum']
).nunique().sort_values('assembly_accession', ascending=False)

femAB_stats['percent'] = femAB_stats.apply(
    lambda row: np.round(
        100 * row['assembly_accession'] / len(gtdb_metadata[gtdb_metadata['gtdb_phylum'] == row.name]),
        1
    ),
    axis=1,
)

femAB_stats = femAB_stats.rename(columns={
    'assembly_accession': '# genomes',
    'percent': '% genomes',
    'gtdb_class': '# GTDB class',
    'gtdb_order': '# GTDB order',
    'gtdb_family': '# GTDB family',
    'gtdb_genus': '# GTDB genus',
})
femAB_stats

,# genomes,# GTDB class,# GTDB order,# GTDB family,# GTDB genus,% genomes
gtdb_phylum,,,,,,
Halobacteriota,133,8,10,20,65,18.0
Nanoarchaeota,57,1,4,27,53,8.2
Thermoproteota,15,5,6,11,15,2.1
Thermoplasmatota,13,3,7,10,11,2.2
Micrarchaeota,9,1,2,5,7,3.7
Aenigmatarchaeota,8,1,4,5,8,6.1
Asgardarchaeota,5,4,4,4,4,2.7
Hydrothermarchaeota,5,1,1,2,3,31.2
Methanobacteriota_B,4,1,2,3,3,7.0


In [51]:
df = summary_df.copy()
df['has_pgh'] = df['pgh_architecture'].notnull().astype(int)
pgh_stats = df[['assembly_accession', 'gtdb_phylum', 'has_pgh']].drop_duplicates('assembly_accession')[['gtdb_phylum', 'has_pgh']].groupby('gtdb_phylum').sum()

femAB_stats_2 = pd.merge(
    femAB_stats,
    pgh_stats,
    on='gtdb_phylum',
    how='inner',
).rename(columns={
    'has_pgh': '# PGH',
})
femAB_stats_2

,# genomes,# GTDB class,# GTDB order,# GTDB family,# GTDB genus,% genomes,# PGH
gtdb_phylum,,,,,,,
Halobacteriota,133,8,10,20,65,18.0,15
Nanoarchaeota,57,1,4,27,53,8.2,5
Thermoproteota,15,5,6,11,15,2.1,1
Thermoplasmatota,13,3,7,10,11,2.2,0
Micrarchaeota,9,1,2,5,7,3.7,0
Aenigmatarchaeota,8,1,4,5,8,6.1,0
Asgardarchaeota,5,4,4,4,4,2.7,1
Hydrothermarchaeota,5,1,1,2,3,31.2,0
Methanobacteriota_B,4,1,2,3,3,7.0,1


In [52]:
df = summary_df.copy()
cat_only_stats = df[
    ['assembly_accession', 'gtdb_phylum', 'has_catalytic_only']
].drop_duplicates('assembly_accession')[['gtdb_phylum', 'has_catalytic_only']].groupby('gtdb_phylum').sum()

femAB_stats_3 = pd.merge(
    femAB_stats_2,
    cat_only_stats,
    on='gtdb_phylum',
    how='inner',
).rename(columns={
    'has_catalytic_only': '# catalytic only',
})
femAB_stats_3

,# genomes,# GTDB class,# GTDB order,# GTDB family,# GTDB genus,% genomes,# PGH,# catalytic only
gtdb_phylum,,,,,,,,
Halobacteriota,133,8,10,20,65,18.0,15,86
Nanoarchaeota,57,1,4,27,53,8.2,5,46
Thermoproteota,15,5,6,11,15,2.1,1,10
Thermoplasmatota,13,3,7,10,11,2.2,0,11
Micrarchaeota,9,1,2,5,7,3.7,0,7
Aenigmatarchaeota,8,1,4,5,8,6.1,0,5
Asgardarchaeota,5,4,4,4,4,2.7,1,5
Hydrothermarchaeota,5,1,1,2,3,31.2,0,0
Methanobacteriota_B,4,1,2,3,3,7.0,1,2


In [65]:
femAB_accessions = set(summary_df['assembly_accession'].unique())
archaeal_pgh_accessions = set(archaeal_pgh['assembly_accession'].unique())

n_no_femAB = len(set(archaeal_accessions) - femAB_accessions)

n_pgh_femAB = len(archaeal_pgh_accessions & femAB_accessions)
n_pgh_no_femAB = len(archaeal_pgh_accessions - femAB_accessions)

print(f'# genomes with FemAB and PGH: {n_pgh_femAB:,} of {len(femAB_accessions):,} ({100 * n_pgh_femAB / len(femAB_accessions):.0f}%)')
print(f'# genomes without FemAB but with PGH: {n_pgh_no_femAB:,} of {n_no_femAB:,} ({100 * n_pgh_no_femAB / n_no_femAB:.0f}%)')

# genomes with FemAB and PGH: 23 of 257 (9%)
# genomes without FemAB but with PGH: 152 of 3,449 (4%)


In [66]:
femAB_accessions = set(summary_df['assembly_accession'].unique())
archaeal_pgh_accessions = set(catalytic_only['assembly_accession'].unique())

n_no_femAB = len(set(archaeal_accessions) - femAB_accessions)

n_pgh_femAB = len(archaeal_pgh_accessions & femAB_accessions)
n_pgh_no_femAB = len(archaeal_pgh_accessions - femAB_accessions)

print(f'# genomes with FemAB and cat only: {n_pgh_femAB:,} of {len(femAB_accessions):,} ({100 * n_pgh_femAB / len(femAB_accessions):.0f}%)')
print(f'# genomes without FemAB but with cat only: {n_pgh_no_femAB:,} of {n_no_femAB:,} ({100 * n_pgh_no_femAB / n_no_femAB:.0f}%)')

# genomes with FemAB and cat only: 177 of 257 (69%)
# genomes without FemAB but with cat only: 1,855 of 3,449 (54%)


In [97]:
femAB_accessions = set(summary_df['assembly_accession'].unique())
archaeal_pgh_accessions = set(m23_only['assembly_accession'].unique())

n_no_femAB = len(set(archaeal_accessions) - femAB_accessions)

n_pgh_femAB = len(archaeal_pgh_accessions & femAB_accessions)
n_pgh_no_femAB = len(archaeal_pgh_accessions - femAB_accessions)

print(f'# genomes with FemAB and M23: {n_pgh_femAB:,} of {len(femAB_accessions):,} ({100 * n_pgh_femAB / len(femAB_accessions):.0f}%)')
print(f'# genomes without FemAB but with M23: {n_pgh_no_femAB:,} of {n_no_femAB:,} ({100 * n_pgh_no_femAB / n_no_femAB:.0f}%)')

# genomes with FemAB and M23: 81 of 257 (32%)
# genomes without FemAB but with M23: 718 of 3,449 (21%)


## PG uptake

Relevant Pfam domain: `SBP_bac_5`.

From [Gilmore and Cava, 2025](https://doi.org/10.1016/j.tim.2024.11.004).

In [95]:
pfam_iterator = pd.read_csv(pfam_path, chunksize=int(1e5), index_col='id')

dfs = []
for chunk_df in pfam_iterator:
    relevant_df = chunk_df[chunk_df['assembly_accession'].isin(archaeal_accessions)]
    relevant_df = relevant_df[relevant_df['hmm_query'].isin(['SBP_bac_5'])]

    if len(relevant_df) > 0:
        dfs.append(relevant_df)

uptake_summary_df = pd.concat(dfs)
uptake_summary_df = pd.merge(
    uptake_summary_df,
    gtdb_metadata.reset_index()[
        ['ncbi_accession', 'domain', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus', 'gtdb_species']
    ].rename(columns={'ncbi_accession': 'assembly_accession'}),
    on='assembly_accession',
    how='left',
)
uptake_summary_df['id'] = uptake_summary_df.apply(lambda row: f"{row['protein_id']}@{row['assembly_accession']}", axis=1)

uptake_summary_df = uptake_summary_df.drop_duplicates('id')

uptake_summary_df = pd.merge(
    uptake_summary_df,
    archaeal_pgh.drop_duplicates('assembly_accession')[['assembly_accession', 'pgh_architecture']],
    on='assembly_accession',
    how='left',
)

uptake_summary_df = pd.merge(
    uptake_summary_df,
    catalytic_only.drop_duplicates('assembly_accession')[['assembly_accession', 'hmm_query']].rename(columns={
        'hmm_query': 'catalytic_only'
    }),
    on='assembly_accession',
    how='left',
)
uptake_summary_df['has_catalytic_only'] = uptake_summary_df['catalytic_only'].notnull().astype(int)
uptake_summary_df = uptake_summary_df.drop(columns=['catalytic_only'])

uptake_summary_df.head()

,assembly_accession,protein_id,hmm_accession,hmm_query,evalue,bitscore,accuracy,start,end,domain,gtdb_phylum,gtdb_class,gtdb_order,gtdb_family,gtdb_genus,gtdb_species,id,pgh_architecture,has_catalytic_only
0,GCA_000016605.1,ABP94618.1,PF00496.25,SBP_bac_5,5.700000e-16,55.5,0.90,43,290,Archaea,Thermoproteota,Thermoprotei_A,Sulfolobales,Sulfolobaceae,Metallosphaera,Metallosphaera sedula,ABP94618.1@GCA_000016605.1,NaN,0
1,GCA_000016605.1,ABP95211.1,PF00496.25,SBP_bac_5,2.400000e-61,204.9,0.92,110,586,Archaea,Thermoproteota,Thermoprotei_A,Sulfolobales,Sulfolobaceae,Metallosphaera,Metallosphaera sedula,ABP95211.1@GCA_000016605.1,NaN,0
2,GCA_000016605.1,ABP96341.1,PF00496.25,SBP_bac_5,4.600000e-55,184.2,0.85,316,769,Archaea,Thermoproteota,Thermoprotei_A,Sulfolobales,Sulfolobaceae,Metallosphaera,Metallosphaera sedula,ABP96341.1@GCA_000016605.1,NaN,0
3,GCA_000145985.1,ADM27384.1,PF00496.25,SBP_bac_5,4.900000e-48,160.9,0.81,77,480,Archaea,Thermoproteota,Thermoprotei_A,Sulfolobales,Ignisphaeraceae,Ignisphaera,Ignisphaera aggregans,ADM27384.1@GCA_000145985.1,NaN,0
4,GCA_000145985.1,ADM27496.1,PF00496.25,SBP_bac_5,8.600000e-97,321.3,0.96,119,482,Archaea,Thermoproteota,Thermoprotei_A,Sulfolobales,Ignisphaeraceae,Ignisphaera,Ignisphaera aggregans,ADM27496.1@GCA_000145985.1,NaN,0


In [96]:
uptake_stats = uptake_summary_df[
    ['gtdb_phylum', 'assembly_accession', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus']
].groupby(
    ['gtdb_phylum']
).nunique().sort_values('assembly_accession', ascending=False)

uptake_stats['percent'] = uptake_stats.apply(
    lambda row: np.round(
        100 * row['assembly_accession'] / len(gtdb_metadata[gtdb_metadata['gtdb_phylum'] == row.name]),
        1
    ),
    axis=1,
)

uptake_stats = uptake_stats.rename(columns={
    'assembly_accession': '# genomes',
    'percent': '% genomes',
    'gtdb_class': '# GTDB class',
    'gtdb_order': '# GTDB order',
    'gtdb_family': '# GTDB family',
    'gtdb_genus': '# GTDB genus',
})
uptake_stats

,# genomes,# GTDB class,# GTDB order,# GTDB family,# GTDB genus,% genomes
gtdb_phylum,,,,,,
Thermoproteota,609,11,27,86,269,86.4
Halobacteriota,545,9,12,37,166,73.7
Thermoplasmatota,307,6,22,40,107,51.3
Asgardarchaeota,156,10,13,23,53,83.0
Methanobacteriota_B,52,1,2,5,12,91.2
Methanobacteriota,48,2,2,3,12,32.0
Hydrothermarchaeota,6,1,1,3,5,37.5
Aenigmatarchaeota,5,1,3,4,5,3.8
Nanoarchaeota,5,1,2,4,4,0.7
